# J2 matin — Préparer des données propres
**9h00–12h30 · Programmation en Python · Mastère 1 Data & IA**

Passer d'un DataFrame brut à une table exploitable, en justifiant ses choix.
Les démonstrations portent sur un table d'employés fictifs ; les exercices sur des ventes réelles (UCI Retail II).

**Objectifs** : Préparer des données propres; Gérer : valeurs manquantes/aberantes, doublons, formats; Encodage de variables categorielles; Transformations simples et complexes; Pipeline de nettoyage.

Ouvrez ce notebook dans Colab puis déposez `sales_dirty.csv` dans le volet **Fichiers** à gauche.
Exécutez les cellules dans l'ordre. Complétez les cellules `#TODO` avant de continuer.
Les fichiers doivent être re-upload après une réinitialisation du runtime.



In [48]:
import pandas as pd

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 16)

## 0. Diagnostic — 9h00
**Démonstration.** Regardons ce que contient cette table d'employés.
`employees_demo_raw` reste notre source : chaque démonstration repart d'une copie.

In [49]:
employees_demo_raw = pd.DataFrame({
    "employee_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 107],
    "name": ["Alice", "Bilal", "Chloé", "Diego", "Emma", "Farid", "Giulia", "Hugo", "Inès", "Jules", "Giulia"],
    "department": ["Sales", "sales", "Sales ", "Engineering", "Engineering", "HR", "HR", "Engineering", "Sales", "Engineering", "HR"],
    "salary": ["42000", "39000", None, "52000", "56000", "41000", "43000", "60000", "45000", None, "43000"],
    "hire_date": ["2022-03-14", "2021-09-01", "2023-01-20", "2020-06-15", "2019-11-04", "2022-07-11", "2021-02-01", "2018-05-30", "2020-10-12", "2024-02-05", "2021-02-01"],
    "office": ["Paris", "Lyon", "Paris", "Remote", None, "Paris", "Lyon", "Remote", "Paris", "Lyon", "Lyon"]
})
employees = employees_demo_raw.copy()
employees

,employee_id,name,department,salary,hire_date,office
0,101,Alice,Sales,42000,2022-03-14,Paris
1,102,Bilal,sales,39000,2021-09-01,Lyon
2,103,Chloé,Sales,None,2023-01-20,Paris
3,104,Diego,Engineering,52000,2020-06-15,Remote
4,105,Emma,Engineering,56000,2019-11-04,None
5,106,Farid,HR,41000,2022-07-11,Paris
6,107,Giulia,HR,43000,2021-02-01,Lyon
7,108,Hugo,Engineering,60000,2018-05-30,Remote
8,109,Inès,Sales,45000,2020-10-12,Paris
9,110,Jules,Engineering,None,2024-02-05,Lyon


In [50]:
employees.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   employee_id  11 non-null     int64 
 1   name         11 non-null     object
 2   department   11 non-null     object
 3   salary       9 non-null      object
 4   hire_date    11 non-null     object
 5   office       10 non-null     object
dtypes: int64(1), object(5)
memory usage: 660.0+ bytes


In [51]:
employees.dtypes

employee_id     int64
name           object
department     object
salary         object
hire_date      object
office         object
dtype: object

In [52]:
employees.isna().sum()

employee_id    0
name           0
department     0
salary         2
hire_date      0
office         1
dtype: int64

In [53]:
employees["department"].value_counts(dropna=False)

Engineering    4
HR             3
Sales          2
sales          1
Sales          1
Name: department, dtype: int64

À l'oral : quelles colonnes méritent une vérification ? Un nombre affiché est-il forcément stocké comme un nombre ?

**À vous — Diagnostic des ventes · 6 min**

Une ligne décrit un produit dans une facture. `quantity` est une quantité, `unit_price_gbp` un prix unitaire en livres sterling. `invoice_no`, `stock_code` et `customer_id` sont des identifiants.

Chargez les données ci-dessous, puis identifiez **au moins trois points à vérifier**. Appuyez chaque observation sur une inspection. Ne modifiez encore aucune valeur.

Source : Daqing Chen, [UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii), CC BY 4.0, feuille 2009–2010. Extrait pédagogique non représentatif ;

In [54]:
sales_dirty = pd.read_csv("sales_dirty.csv")
sales = sales_dirty.copy()

In [55]:
# TODO
sales.info()
sales.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202 entries, 0 to 201
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_no      202 non-null    object 
 1   stock_code      202 non-null    object 
 2   description     202 non-null    object 
 3   quantity        202 non-null    int64  
 4   invoice_date    202 non-null    object 
 5   unit_price_gbp  202 non-null    float64
 6   customer_id     172 non-null    float64
 7   country         200 non-null    object 
dtypes: float64(2), int64(1), object(5)
memory usage: 12.8+ KB


,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,UNITED KINGDOM
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [56]:
# TODO 
sales.isna().sum()

invoice_no         0
stock_code         0
description        0
quantity           0
invoice_date       0
unit_price_gbp     0
customer_id       30
country            2
dtype: int64

In [57]:
# TODO faire un comptage sur des valeurs catégoriques numériques longues 'customers'
# sort des bills et des histo qui feront ramer la machine
sales["country"].value_counts(dropna=False)

United Kingdom    149
France             18
Germany            15
Netherlands        15
NaN                 2
UNITED KINGDOM      1
France              1
france              1
Name: country, dtype: int64

In [58]:
# TODO la méthodde unique values() est plus rapide que 
# value_counts() pour compter les valeurs uniques
sales["country"].unique()


array(['UNITED KINGDOM', 'United Kingdom', nan, 'France ', 'france',
       'France', 'Germany', 'Netherlands'], dtype=object)

In [59]:
# TODO pour obtenir des infos statistiques 
# sur les colonnes numériques, 
# on peut utiliser la méthode describe()
sales.describe()

,quantity,unit_price_gbp,customer_id
count,202.000000,202.000000,172.000000
mean,24.688119,2.972673,14607.976744
std,57.812329,2.417648,1955.474878
min,-6.000000,0.420000,12490.000000
25%,3.000000,1.250000,13078.000000
50%,12.000000,2.275000,13635.000000
75%,24.000000,4.250000,15362.000000
max,480.000000,18.000000,18102.000000


In [60]:
# TODO Les valeurs négatives 
# dans les colonnes numériques sont souvent des erreurs de saisie.
sales[sales["quantity"] < 0]

,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country
119,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom
120,C489459,90200D,PINK SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom
121,C489459,90200B,BLACK SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom
122,C489459,90200E,GREEN SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom
123,C489459,90200C,BLUE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom
...,...,...,...,...,...,...,...,...
134,C489503,21533,RETRO SPOT LARGE MILK JUG,-1,2009-12-01 11:04:00,4.95,16011.0,United Kingdom
135,C489504,85083,KISS REINDEER SCANDINAVIAN STOCKING,-6,2009-12-01 11:10:00,2.55,13916.0,United Kingdom
136,C489518,20892,SET/3 TALL GLASS CANDLE HOLDER PINK,-2,2009-12-01 11:35:00,12.75,15461.0,United Kingdom
137,C489518,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,-1,2009-12-01 11:35:00,7.95,15461.0,United Kingdom


In [61]:
# TODO 


In [62]:
# TODO


Vos observations :

1 ->

2 ->

3 ->

## 1. Valeurs manquantes — 9h15
**Démonstration.** Une absence peut conduire à conserver, exclure ou remplacer une valeur, selon l'objectif.

In [63]:
employees.isna().sum()

employee_id    0
name           0
department     0
salary         2
hire_date      0
office         1
dtype: int64

In [64]:
# on va voir de plus près -> office 
employees.loc[employees["office"].isna()]

,employee_id,name,department,salary,hire_date,office
4,105,Emma,Engineering,56000,2019-11-04,None


In [65]:
# on va voir de plus près -> salary
employees.loc[employees["salary"].isna()]

,employee_id,name,department,salary,hire_date,office
2,103,Chloé,Sales,None,2023-01-20,Paris
9,110,Jules,Engineering,None,2024-02-05,Lyon


Cas de office (simple) -> Nouvelle catégorie "Unknown"

In [66]:
# on peut remplacer les valeurs manquantes 
# par une valeur par défaut . Ici on a une série office remplacée par la série unknown
employees["office"] = employees["office"].fillna("Unknown")

Cas de salary (moins simple) -> imputer la mediane

In [67]:
median_salary = employees["salary"].median()

# OUPS ?

In [68]:
# et oui !
employees.dtypes

employee_id     int64
name           object
department     object
salary         object
hire_date      object
office         object
dtype: object

Pour calculer sur `salary`, convertissons d'abord le texte en nombres. Ici les valeurs renseignées sont valides ; une valeur invalide provoquerait une erreur au lieu d'être effacée silencieusement.

In [69]:
employees["salary"] = pd.to_numeric(employees["salary"])


In [70]:
median_salary = employees["salary"].median()
median_salary

43000.0

In [71]:
employees["salary"] = employees["salary"].fillna(median_salary)

employees

,employee_id,name,department,salary,hire_date,office
0,101,Alice,Sales,42000.0,2022-03-14,Paris
1,102,Bilal,sales,39000.0,2021-09-01,Lyon
2,103,Chloé,Sales,43000.0,2023-01-20,Paris
3,104,Diego,Engineering,52000.0,2020-06-15,Remote
4,105,Emma,Engineering,56000.0,2019-11-04,Unknown
5,106,Farid,HR,41000.0,2022-07-11,Paris
6,107,Giulia,HR,43000.0,2021-02-01,Lyon
7,108,Hugo,Engineering,60000.0,2018-05-30,Remote
8,109,Inès,Sales,45000.0,2020-10-12,Paris
9,110,Jules,Engineering,43000.0,2024-02-05,Lyon


Imputer signifie remplacer une valeur absente par une estimation. La médiane est la valeur centrale des valeurs connues triées ; Pandas ignore ici les `nan` pour la calculer.
Cette estimation n'est pas un salaire miraculeusement retrouvé : elle peut servir à une expérimentation mais pas à établir une fiche de paie.

**Exercice A — Conserver pour quel usage ? · 18 min**

1. Comptez les absences dans `sales`, puis affichez quelques lignes sans `customer_id`.
2. Créez `sales_with_customer`, une table séparée dédiée à une analyse par client, sans client manquant. Comptez les lignes exclues.
3. Dans `sales`, remplacez les pays absents par une valeur qui vous semble pertinente.
4. Pourquoi serait-il incorrect de supprimer automatiquement toute ligne contenant une valeur manquante ?

In [72]:
# TODO A1 pour compter les absences dans la colonne sales, 
# on peut utiliser la méthode isna() suivie de sum() et aussi sans afficher la colonne customer_id


sales_dirty = pd.read_csv("sales_dirty.csv")
sales = sales_dirty.copy()
absences = sales["customer_id"].isna().sum()
print(f"Nombre de customer_id absents : {absences}")

# Afficher les lignes sans des customer_id absents
display(sales[sales["customer_id"].isna()])


Nombre de customer_id absents : 30


,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country
139,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
140,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
156,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
157,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
158,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
179,489548,21559,STRAWBERRY LUNCHBOX WITH CUTLERY,2,2009-12-01 12:32:00,2.55,NaN,United Kingdom
180,489548,21558,SKULL LUNCHBOX WITH CUTLERY,3,2009-12-01 12:32:00,2.55,NaN,United Kingdom
181,489548,22352,LUNCHBOX WITH CUTLERY RETROSPOT,1,2009-12-01 12:32:00,2.55,NaN,United Kingdom
182,489548,21004,ROSE DU SUD CHILDS APRON,3,2009-12-01 12:32:00,4.25,NaN,United Kingdom


In [73]:
# TODO A1 
sales[sales["customer_id"].isna()]

,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country
139,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
140,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
156,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
157,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
158,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
179,489548,21559,STRAWBERRY LUNCHBOX WITH CUTLERY,2,2009-12-01 12:32:00,2.55,NaN,United Kingdom
180,489548,21558,SKULL LUNCHBOX WITH CUTLERY,3,2009-12-01 12:32:00,2.55,NaN,United Kingdom
181,489548,22352,LUNCHBOX WITH CUTLERY RETROSPOT,1,2009-12-01 12:32:00,2.55,NaN,United Kingdom
182,489548,21004,ROSE DU SUD CHILDS APRON,3,2009-12-01 12:32:00,4.25,NaN,United Kingdom


In [74]:
# TODO A2 pour créer un nouveau DataFrame sales_with_customer qui ne contient que les lignes avec des customer_id présents,
#  on peut utiliser la méthode dropna() sur la colonne customer_id
sales_with_customer = sales.dropna(subset=["customer_id"])
sales_with_customer

,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,UNITED KINGDOM
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
197,489889,21681,GIANT MEDINA STAMPED METAL BOWL,24,2009-12-02 16:52:00,8.50,14646.0,Netherlands
198,489889,21841,BABY MOUSE RED GINGHAM DRESS,36,2009-12-02 16:52:00,3.75,14646.0,Netherlands
199,489889,21531,RETRO SPOT SUGAR JAM BOWL,48,2009-12-02 16:52:00,2.10,14646.0,Netherlands
200,489436,22109,FULL ENGLISH BREAKFAST PLATE,16,2009-12-01 09:06:00,3.39,13078.0,United Kingdom


In [75]:
#TODO A2
#sans les lignes exclus
len(sales) - len(sales_with_customer)

30

In [148]:
# TODO A3 on impute par must frequent strategy en machine learning
sales["country"] = sales["country"].fillna("Unknown")

In [77]:
# TEST YOUR CODE
assert sales["country"].notna().all()
assert len(sales) == len(sales_dirty)
assert sales_with_customer["customer_id"].notna().all()

**Remarque.** Une vente sans client identifié reste exploitable par produit ou par pays. L'exclusion est justifiée pour une analyse individuelle des clients, pas pour toute analyse des ventes. `Unknown` indique une information géographique inconnue sans inventer un pays.

*Indice A :* sélectionnez les lignes avec un masque `isna()`. Pour exclure selon une seule colonne, utilisez le paramètre `subset`.

## 2. Doublons, types et dates — 9h55
**Démonstration.** `duplicated()` marque les copies après la première occurrence ; `keep=False` permet de voir toutes les occurrences concernées.

In [78]:
employees

,employee_id,name,department,salary,hire_date,office
0,101,Alice,Sales,42000.0,2022-03-14,Paris
1,102,Bilal,sales,39000.0,2021-09-01,Lyon
2,103,Chloé,Sales,43000.0,2023-01-20,Paris
3,104,Diego,Engineering,52000.0,2020-06-15,Remote
4,105,Emma,Engineering,56000.0,2019-11-04,Unknown
5,106,Farid,HR,41000.0,2022-07-11,Paris
6,107,Giulia,HR,43000.0,2021-02-01,Lyon
7,108,Hugo,Engineering,60000.0,2018-05-30,Remote
8,109,Inès,Sales,45000.0,2020-10-12,Paris
9,110,Jules,Engineering,43000.0,2024-02-05,Lyon


In [79]:
employees.duplicated().sum()

1

In [80]:
employees.drop_duplicates(inplace=True)

In [47]:
employees.loc[employees.duplicated(keep=False)]

,employee_id,name,department,salary,hire_date,office
6,107,Giulia,HR,43000.0,2021-02-01,Lyon
10,107,Giulia,HR,43000.0,2021-02-01,Lyon


In [81]:
print("Avant  :", len(employees))
employees = employees.drop_duplicates().copy()
print("Après :", len(employees))

Avant  : 10
Après : 10


In [82]:
# rappel -> on avait déjà changé de type de "salary" pour calculer la mediane
employees.dtypes

employee_id      int64
name            object
department      object
salary         float64
hire_date       object
office          object
dtype: object

In [83]:
# convertir en date
employees["hire_date"] = pd.to_datetime(employees["hire_date"])

# extraire année et mois dans 2 nouvelles colonnes
employees["hire_year"] = employees["hire_date"].dt.year
employees["hire_month"] = employees["hire_date"].dt.month

display(employees[["name", "salary", "hire_date", "hire_year", "hire_month"]])
display(employees.dtypes)

,name,salary,hire_date,hire_year,hire_month
0,Alice,42000.0,2022-03-14,2022,3
1,Bilal,39000.0,2021-09-01,2021,9
2,Chloé,43000.0,2023-01-20,2023,1
3,Diego,52000.0,2020-06-15,2020,6
4,Emma,56000.0,2019-11-04,2019,11
5,Farid,41000.0,2022-07-11,2022,7
6,Giulia,43000.0,2021-02-01,2021,2
7,Hugo,60000.0,2018-05-30,2018,5
8,Inès,45000.0,2020-10-12,2020,10
9,Jules,43000.0,2024-02-05,2024,2


employee_id             int64
name                   object
department             object
salary                float64
hire_date      datetime64[ns]
office                 object
hire_year               int64
hire_month              int64
dtype: object

**Exercice B — Nettoyer sans perdre les produits d'une facture · 16 min**

Sur `sales`, après A :

1. Comptez les copies exactes en trop, puis affichez toutes les occurrences concernées.
2. Supprimez les copies exactes.
3. Comptez les lignes présentant les mêmes `invoice_no` (`keep=False`). Est-ce inquiétant ? Diagnostiquez
4. Convertissez `customer_id` en entier (`"Int64"`), `invoice_date` en datetime, puis créez `invoice_hour` avec l'heure d'achat.

Pourquoi ne faut-il pas dédupliquer sur `invoice_no` seul ?

*Indice :* `duplicated(subset=[...], keep=False)` compare uniquement les colonnes choisies.

In [104]:
# TODO B1 compter les copies dupliquées et afficher toutes les lignes dupliquées dans le DataFrame sales

sales.duplicated().sum()

0

In [105]:
sales.loc[sales.duplicated(keep=False)]

,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country


In [116]:
# TODO B2
sales = sales_dirty.copy().drop_duplicates()
sales

,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,UNITED KINGDOM
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
195,489889,85123A,WHITE HANGING HEART T-LIGHT HOLDER,480,2009-12-02 16:52:00,2.55,14646.0,Netherlands
196,489889,22336,DOVE DECORATION PAINTED ZINC,288,2009-12-02 16:52:00,0.55,14646.0,Netherlands
197,489889,21681,GIANT MEDINA STAMPED METAL BOWL,24,2009-12-02 16:52:00,8.50,14646.0,Netherlands
198,489889,21841,BABY MOUSE RED GINGHAM DRESS,36,2009-12-02 16:52:00,3.75,14646.0,Netherlands


In [118]:
# TODO B3 compter les lignes avec les mêmes invoice_no et afficher toutes les lignes avec des invoice_no dupliqués dans le DataFrame sales

sales.duplicated(subset=["invoice_no"], keep=False).sum()

197

In [ ]:
# TODO B3


In [119]:
# TODO B4 Convertir customer_id en type int64, et convertir invoice_date en type date
sales["customer_id"] = sales["customer_id"].astype("Int64")
sales["invoice_date"] = pd.to_datetime(sales["invoice_date"])
invoice_hour = sales["invoice_date"].dt.hour

sales

,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,UNITED KINGDOM
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom
...,...,...,...,...,...,...,...,...
195,489889,85123A,WHITE HANGING HEART T-LIGHT HOLDER,480,2009-12-02 16:52:00,2.55,14646,Netherlands
196,489889,22336,DOVE DECORATION PAINTED ZINC,288,2009-12-02 16:52:00,0.55,14646,Netherlands
197,489889,21681,GIANT MEDINA STAMPED METAL BOWL,24,2009-12-02 16:52:00,8.50,14646,Netherlands
198,489889,21841,BABY MOUSE RED GINGHAM DRESS,36,2009-12-02 16:52:00,3.75,14646,Netherlands


**Correction.** Deux copies exactes sont supprimées : 202 → 200 lignes. Une facture peut porter plusieurs produits. Dédupliquer son numéro seul ferait disparaître des lignes utiles et n'aurait pas vraiment de sens.

**Pause — 10h30–10h45**

## 3. Catégories et transformations — 10h45
**Démonstration.** Harmoniser les départements, puis leur associer une function (Métier vs Transverse).
`replace` conserve les valeurs non mentionnées. Avec un dictionnaire, `map` produit un NaN pour une valeur sans correspondance : vérifiez toujours la couverture.

In [120]:
employees

,employee_id,name,department,salary,hire_date,office,hire_year,hire_month
0,101,Alice,Sales,42000.0,2022-03-14,Paris,2022,3
1,102,Bilal,sales,39000.0,2021-09-01,Lyon,2021,9
2,103,Chloé,Sales,43000.0,2023-01-20,Paris,2023,1
3,104,Diego,Engineering,52000.0,2020-06-15,Remote,2020,6
4,105,Emma,Engineering,56000.0,2019-11-04,Unknown,2019,11
5,106,Farid,HR,41000.0,2022-07-11,Paris,2022,7
6,107,Giulia,HR,43000.0,2021-02-01,Lyon,2021,2
7,108,Hugo,Engineering,60000.0,2018-05-30,Remote,2018,5
8,109,Inès,Sales,45000.0,2020-10-12,Paris,2020,10
9,110,Jules,Engineering,43000.0,2024-02-05,Lyon,2024,2


In [121]:
employees["department"] = employees["department"].replace({"sales": "Sales"})

employees

,employee_id,name,department,salary,hire_date,office,hire_year,hire_month
0,101,Alice,Sales,42000.0,2022-03-14,Paris,2022,3
1,102,Bilal,Sales,39000.0,2021-09-01,Lyon,2021,9
2,103,Chloé,Sales,43000.0,2023-01-20,Paris,2023,1
3,104,Diego,Engineering,52000.0,2020-06-15,Remote,2020,6
4,105,Emma,Engineering,56000.0,2019-11-04,Unknown,2019,11
5,106,Farid,HR,41000.0,2022-07-11,Paris,2022,7
6,107,Giulia,HR,43000.0,2021-02-01,Lyon,2021,2
7,108,Hugo,Engineering,60000.0,2018-05-30,Remote,2018,5
8,109,Inès,Sales,45000.0,2020-10-12,Paris,2020,10
9,110,Jules,Engineering,43000.0,2024-02-05,Lyon,2024,2


In [122]:
functions = {
    "Sales": "Support",
    "Engineering": "Core Business",
    "HR": "Support"}

employees["function"] = employees["department"].map(functions)

employees # OUPS

,employee_id,name,department,salary,hire_date,office,hire_year,hire_month,function
0,101,Alice,Sales,42000.0,2022-03-14,Paris,2022,3,Support
1,102,Bilal,Sales,39000.0,2021-09-01,Lyon,2021,9,Support
2,103,Chloé,Sales,43000.0,2023-01-20,Paris,2023,1,NaN
3,104,Diego,Engineering,52000.0,2020-06-15,Remote,2020,6,Core Business
4,105,Emma,Engineering,56000.0,2019-11-04,Unknown,2019,11,Core Business
5,106,Farid,HR,41000.0,2022-07-11,Paris,2022,7,Support
6,107,Giulia,HR,43000.0,2021-02-01,Lyon,2021,2,Support
7,108,Hugo,Engineering,60000.0,2018-05-30,Remote,2018,5,Core Business
8,109,Inès,Sales,45000.0,2020-10-12,Paris,2020,10,Support
9,110,Jules,Engineering,43000.0,2024-02-05,Lyon,2024,2,Core Business


In [123]:
#pour enlever les espaces dans les colonnes, on peut utiliser .strip() sur les colonnes de type object
"sales".strip("s")

'ale'

In [124]:
employees['department'].unique()

array(['Sales', 'Sales ', 'Engineering', 'HR'], dtype=object)

In [125]:
# on appelle str.strip() methode de classe str Python natif
employees['department'] = employees['department'].str.strip()

# on remape
employees["function"] = employees["department"].map(functions)

employees # et voilà !

,employee_id,name,department,salary,hire_date,office,hire_year,hire_month,function
0,101,Alice,Sales,42000.0,2022-03-14,Paris,2022,3,Support
1,102,Bilal,Sales,39000.0,2021-09-01,Lyon,2021,9,Support
2,103,Chloé,Sales,43000.0,2023-01-20,Paris,2023,1,Support
3,104,Diego,Engineering,52000.0,2020-06-15,Remote,2020,6,Core Business
4,105,Emma,Engineering,56000.0,2019-11-04,Unknown,2019,11,Core Business
5,106,Farid,HR,41000.0,2022-07-11,Paris,2022,7,Support
6,107,Giulia,HR,43000.0,2021-02-01,Lyon,2021,2,Support
7,108,Hugo,Engineering,60000.0,2018-05-30,Remote,2018,5,Core Business
8,109,Inès,Sales,45000.0,2020-10-12,Paris,2020,10,Support
9,110,Jules,Engineering,43000.0,2024-02-05,Lyon,2024,2,Core Business


In [126]:
# encodons department avec get_dummies permet d'encoder des variables catégoriques
department_encoded = pd.get_dummies(employees["department"], prefix="department", dtype=int)
department_encoded

# concat sera le sujet de cet aprèms midi (no spoil!)

,department_Engineering,department_HR,department_Sales
0,0,0,1
1,0,0,1
2,0,0,1
3,1,0,0
4,1,0,0
5,0,1,0
6,0,1,0
7,1,0,0
8,0,0,1
9,1,0,0


Chaque colonne indicatrice vaut 1 si la catégorie est présente, 0 sinon. Cela permet une représentation numérique pour un futur modèle, sans établir d'ordre entre les catégories.

**Démonstration — Choisir entre vectorisation et `apply`.** Une hausse de salaire de 10 % se calcule directement sur la colonne.

In [127]:
employees["salary_plus_10pct"] = employees["salary"] * 1.10

employees[["name", "salary", "salary_plus_10pct"]]

,name,salary,salary_plus_10pct
0,Alice,42000.0,46200.0
1,Bilal,39000.0,42900.0
2,Chloé,43000.0,47300.0
3,Diego,52000.0,57200.0
4,Emma,56000.0,61600.0
5,Farid,41000.0,45100.0
6,Giulia,43000.0,47300.0
7,Hugo,60000.0,66000.0
8,Inès,45000.0,49500.0
9,Jules,43000.0,47300.0


In [128]:
employees

,employee_id,name,department,salary,hire_date,office,hire_year,hire_month,function,salary_plus_10pct
0,101,Alice,Sales,42000.0,2022-03-14,Paris,2022,3,Support,46200.0
1,102,Bilal,Sales,39000.0,2021-09-01,Lyon,2021,9,Support,42900.0
2,103,Chloé,Sales,43000.0,2023-01-20,Paris,2023,1,Support,47300.0
3,104,Diego,Engineering,52000.0,2020-06-15,Remote,2020,6,Core Business,57200.0
4,105,Emma,Engineering,56000.0,2019-11-04,Unknown,2019,11,Core Business,61600.0
5,106,Farid,HR,41000.0,2022-07-11,Paris,2022,7,Support,45100.0
6,107,Giulia,HR,43000.0,2021-02-01,Lyon,2021,2,Support,47300.0
7,108,Hugo,Engineering,60000.0,2018-05-30,Remote,2018,5,Core Business,66000.0
8,109,Inès,Sales,45000.0,2020-10-12,Paris,2020,10,Support,49500.0
9,110,Jules,Engineering,43000.0,2024-02-05,Lyon,2024,2,Core Business,47300.0


Pour une petite règle à plusieurs informations, une fonction sur chaque ligne peut être plus lisible.  on utilise `apply()` avec `axis=1`, `row` représente une ligne ; l'ordre des conditions fixe leur priorité (return dans chaque if).

In [129]:
def calculate_bonus(row):
    if pd.isna(row["salary"]):
        return 0

    # remarquez que si on rentre dans un "if" on en ressort jamais ici (pas besoin de "else" du coup)
    if row["department"] == "Sales" and row["salary_plus_10pct"] <= 45000:
        return row["salary"] * 0.10
    if row['office'] == "Remote":
        return row["salary"] * 0.05
    return row["salary"] * 0.03

employees["bonus"] = employees.apply(
    calculate_bonus,
    axis=1
)
employees[
        ["name", "salary", "department", "office", "salary_plus_10pct", "bonus"]
    ]

,name,salary,department,office,salary_plus_10pct,bonus
0,Alice,42000.0,Sales,Paris,46200.0,1260.0
1,Bilal,39000.0,Sales,Lyon,42900.0,3900.0
2,Chloé,43000.0,Sales,Paris,47300.0,1290.0
3,Diego,52000.0,Engineering,Remote,57200.0,2600.0
4,Emma,56000.0,Engineering,Unknown,61600.0,1680.0
5,Farid,41000.0,HR,Paris,45100.0,1230.0
6,Giulia,43000.0,HR,Lyon,47300.0,1290.0
7,Hugo,60000.0,Engineering,Remote,66000.0,3000.0
8,Inès,45000.0,Sales,Paris,49500.0,1350.0
9,Jules,43000.0,Engineering,Lyon,47300.0,1290.0


In [130]:
def test(x):
    return x ** 2

type(test), type(lambda x: x ** 2)

(function, function)

**Exercice C — Harmoniser puis transformer · 18 min**

Continuez sur `sales` après B.

1. Inspectez les pays. Corrigez les variantes avec `replace` et/ou autre méthode adaptée.
2. Avec `map`, créez `market` selon la table ci-dessous. Vérifiez qu'aucun pays ne reste sans règle et qu'aucun NaN n'est créé.
3. Créez une table séparée `market_encoded` avec les indicatrices 0/1 de `market` et le préfixe `market`.

| Pays harmonisé | market |
|---|---|
| United Kingdom | UK |
| France | EU |
| Germany | EU |
| Netherlands | EU |
| Unknown | Unknown |

Ces règles couvrent les pays de cet extrait. `market` est un regroupement commercial pour l'exercice.

4. Écrivez `transaction_type(row)` selon les priorités suivantes : quantité négative → `"return"` ; sinon, client id inexistant → `"sale_without_customer"` ; sinon → `"identified_sale"`.
5. Appliquez cette fonction ligne par ligne pour créer `transaction_type`. Inspectez les effectifs et un exemple de chaque catégorie.
6. Pourquoi ne doit-on pas utiliser `apply` pour calculer simplement `quantity * unit_price_gbp` ? Créez la colonne `total_price` qui contient cette valeur (avec la bonne méthode)

*Indice :* `pd.isna(...)` fonctionne aussi avec les valeurs absentes d'une colonne `Int64`. Vérifiez les pays harmonisés avant le mapping.

In [151]:
# TODO C1 corrigeons les variantes des pays avec la méthode replace() de pandas sur la colonne country du DataFrame sales
sales["country"].unique()

array(['United Kingdom', 'Unknown', 'France', 'france', 'Germany',
       'Netherlands'], dtype=object)

In [152]:
sales["country"].str.strip().str.capitalize().value_counts()

United kingdom    148
France             20
Germany            15
Netherlands        15
Unknown             2
Name: country, dtype: int64

In [153]:
sales["country"] = sales["country"].str.strip().replace({
    "France": "FR",
    "UNITED KINGDOM": "United Kingdom",
})


In [154]:
sales["country"].unique()

array(['United Kingdom', 'Unknown', 'FR', 'france', 'Germany',
       'Netherlands'], dtype=object)

In [156]:
# TODO C2

market_mapping = {
    "United Kingdom": "UK",
    "France": "EU",
    "Germany": "EU",
    "Netherlands": "EU",
    "Unknown": "Unknown",
}


sales["market"] = sales["country"].map(market_mapping)

sales.sample(5)


,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country,market
145,489526,84946,ANTIQUE SILVER TEA GLASS ETCHED,12,2009-12-01 11:50:00,1.25,12533,Germany,EU
34,489437,21364,PEACE SMALL WOOD LETTERS,2,2009-12-01 09:08:00,6.75,15362,United Kingdom,UK
119,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592,United Kingdom,UK
196,489889,22336,DOVE DECORATION PAINTED ZINC,288,2009-12-02 16:52:00,0.55,14646,Netherlands,EU
163,489548,22147,FELTCRAFT BUTTERFLY HEARTS,2,2009-12-01 12:32:00,1.45,<NA>,United Kingdom,UK


In [158]:
# TODO C3
marked_encoded = pd.get_dummies(sales["market"], prefix="market", dtype=int)
marked_encoded

,market_EU,market_UK,market_Unknown
0,0,1,0
1,0,1,0
2,0,1,0
3,0,1,0
4,0,1,0
...,...,...,...
195,1,0,0
196,1,0,0
197,1,0,0
198,1,0,0


In [170]:
# TODO C4 Écrivez `transaction_type(row)` selon les priorités suivantes : quantité négative → `"return"` ; 
# sinon, client id inexistant → `"sale_without_customer"` ;
# sinon → `"identified_sale"`.

def transaction_type(row):
    if row["quantity"] < 0:
        return "return"
    if pd.isna(row["customer_id"]):
        return "sale_without_customer"
    return "identified_sale"


In [172]:
# TODO C5  Appliquez cette fonction ligne par ligne pour créer `transaction_type`. 
# Inspectez les effectifs et un exemple de chaque catégorie.
transaction_types = sales.apply(transaction_type, axis=1)



# montre un entrée pour chaque type de transaction
sales["transaction_type"] = sales.apply(transaction_type, axis=1)

print(sales["transaction_type"].value_counts())

for category in sales["transaction_type"].unique():
    display(sales.loc[sales["transaction_type"] == category].head(1))

identified_sale          150
sale_without_customer     30
return                    20
Name: transaction_type, dtype: int64


,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country,market,transaction_type
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,UK,identified_sale


,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country,market,transaction_type
119,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592,United Kingdom,UK,return


,invoice_no,stock_code,description,quantity,invoice_date,unit_price_gbp,customer_id,country,market,transaction_type
139,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,<NA>,United Kingdom,UK,sale_without_customer


In [175]:
# TODO C6 Pourquoi ne doit-on pas utiliser `apply` pour calculer simplement `quantity * unit_price_gbp` ? 
# Créez la colonne `total_price` qui contient cette valeur (avec la bonne méthode)
sales["total_price"] = sales["quantity"] * sales["unit_price_gbp"]

sales["total_price"] = sales.apply(lambda row: row["quantity"] * row["unit_price_gbp"], axis=1)

## 4. Method chaining — 11h25
**Démonstration.** Les opérations connues s'enchaînent entre parenthèses. `assign` crée ou remplace une colonne ; la fonction `lambda df:` reçoit la table à cette étape.
La source reste intacte.

In [ ]:
employees_2 = employees_demo_raw.copy()

employees_clean = (
    employees_2
    .drop_duplicates()
    .assign(
        salary=lambda df: pd.to_numeric(df["salary"]),
        office=lambda df: df["office"].fillna("Unknown"),
        hire_date=lambda df: pd.to_datetime(df["hire_date"])
    )
)
employees_clean

## 5. TP final — 11h40 · 35 min
L'équipe prépare une future analyse des ventes **par produit, date et zone géographique**.
À partir de la nouvelle lecture ci-dessous, produisez `sales_ready`. Choisissez les transformations utiles et justifiez vos décisions.

**Résultat attendu**

- Une ligne reste une ligne de facture : seules les copies exactes sont retirées.
- Les dates sont exploitables en datetime et les colonnes utiles ont des types cohérents. `customer_id` est un entier nullable ; les codes de facture et de produit restent des identifiants.
- Les pays sont harmonisés ; l'absence de pays est représentée explicitement par `Unknown`.
- `market` respecte les correspondances définies en C, sans NaN involontaire.
- Les transactions sans client identifié restent disponibles pour l'objectif annoncé. Les quantités négatives sont conservées pour tenir compte des retours.
- Aucune transformation n'introduit de valeur manquante involontaire.

Vous choisissez les outils et l'ordre des opérations. Les contrôles ci-dessous décrivent une partie du résultat attendu ; complétez-les par vos inspections.

In [135]:
sales_tp = pd.read_csv("sales_dirty.csv")

In [ ]:
# TODO TP


**Contrôles — à exécuter après votre préparation.** Un contrôle en échec est un point à examiner dans vos choix ou votre code.

In [ ]:
assert not sales_ready.duplicated().any(), "Il reste des copies exactes"
assert len(sales_ready) == len(sales_tp.drop_duplicates()), "Des lignes utiles ont disparu"
assert pd.api.types.is_datetime64_any_dtype(sales_ready["invoice_date"])
assert str(sales_ready["customer_id"].dtype) == "Int64"
assert pd.api.types.is_numeric_dtype(sales_ready["quantity"])
assert pd.api.types.is_numeric_dtype(sales_ready["unit_price_gbp"])
assert sales_ready["country"].notna().all()
assert sales_ready["market"].notna().all()
assert sales_ready["country"].isin([
    "United Kingdom", "France", "Germany", "Netherlands", "Unknown"
]).all()
assert sales_ready.loc[sales_ready["country"] == "United Kingdom", "market"].eq("UK").all()
assert sales_ready.loc[sales_ready["country"].isin(["France", "Germany", "Netherlands"]), "market"].eq("EU").all()
assert sales_ready.loc[sales_ready["country"] == "Unknown", "market"].eq("Unknown").all()
assert sales_ready["customer_id"].isna().sum() == sales_tp.drop_duplicates()["customer_id"].isna().sum()
assert not sales_ready.drop(columns="customer_id").isna().any().any()
assert (sales_ready["quantity"] < 0).sum() == (sales_tp.drop_duplicates()["quantity"] < 0).sum()
print("Contrôles du TP réussis")

**Justifiez vos choix**

1. Quelles transformations vues ce matin avez-vous utilisées ? Pourquoi ?
2. Quelles transformations vues ce matin n'étaient pas nécessaires ici ? Pourquoi ?
3. Pourquoi ne pas supprimer automatiquement les lignes sans `customer_id` ?
4. Pourquoi plusieurs lignes partageant `invoice_no` peuvent-elles être conservées ?

Vos réponses : …

*Bonus :* proposez une table séparée `sales_customer_ready` destinée à une analyse par client. Quel choix change ?

In [ ]:
# TODO TP BONUS

BRAVO 👏